# 00 — Explore featurization

Phase 0 walkthrough of `tantalus.featurize`: drawing molecules, canonical SMILES,
atom/bond features, the PyG graph, Morgan fingerprints, and Tanimoto similarity.
No dataset needed. Every molecule here is a hand-picked, well-known tastant.

In [ ]:
import numpy as np
from rdkit import Chem, DataStructs
from rdkit.Chem import Draw, rdFingerprintGenerator

from tantalus import featurize as fz

MOLECULES = {
    "sucrose (sweet)": "OC[C@H]1O[C@@](CO)(O[C@H]2O[C@H](CO)[C@@H](O)[C@H](O)[C@H]2O)[C@@H](O)[C@@H]1O",
    "sucralose (sweet)": "OC[C@H]1O[C@H](O[C@]2(CCl)O[C@H](CCl)[C@@H](O)[C@@H]2O)[C@H](O)[C@@H](Cl)[C@H]1O",
    "aspartame (sweet)": "COC(=O)[C@H](Cc1ccccc1)NC(=O)[C@@H](N)CC(=O)O",
    "caffeine (bitter)": "Cn1cnc2c1c(=O)n(C)c(=O)n2C",
    "quinine (bitter)": "COc1ccc2nccc([C@@H](O)[C@H]3C[C@@H]4CC[N@]3C[C@@H]4C=C)c2c1",
    "glutamic acid (umami)": "N[C@@H](CCC(=O)O)C(=O)O",
}

## 1. Draw the molecules

In [ ]:
mols = [Chem.MolFromSmiles(s) for s in MOLECULES.values()]
Draw.MolsToGridImage(mols, molsPerRow=3, subImgSize=(260, 220), legends=list(MOLECULES))

## 2. Canonicalization

The same molecule can be written as many SMILES strings. `canonicalize` maps them all to
one string, which is why we canonicalize **before** any dedup, merge or split. If we didn't,
a molecule could sit in both train and test under two spellings (leakage).

In [ ]:
ethanol_spellings = ["CCO", "OCC", "C(O)C", "[CH3][CH2][OH]"]
for s in ethanol_spellings:
    print(f"{s:>16} -> {fz.canonicalize(s)}")
assert len({fz.canonicalize(s) for s in ethanol_spellings}) == 1

# Invalid input returns None instead of raising
print(
    "invalid:",
    fz.canonicalize("not_a_smiles"),
    fz.canonicalize(""),
    fz.canonicalize(None),
)

## 3. Atom features

Each atom becomes a 33-dim vector: one-hots (each with an "other" slot) for element,
degree, formal charge, hybridization and total H, plus aromatic and in-ring flags.
Below we decode caffeine's first atom (a methyl carbon on N) back into readable fields.

In [ ]:
BLOCKS = [
    ("element", fz.ELEMENTS + ["other"]),
    ("degree", fz.DEGREES + ["other"]),
    ("formal_charge", fz.FORMAL_CHARGES + ["other"]),
    ("hybridization", [str(h) for h in fz.HYBRIDIZATIONS] + ["other"]),
    ("total_H", fz.NUM_HS + ["other"]),
]


def decode_atom(vec: list[float]) -> dict:
    """Turn an atom feature vector back into readable field values."""
    out, i = {}, 0
    for name, choices in BLOCKS:
        block = vec[i : i + len(choices)]
        out[name] = choices[block.index(1.0)]
        i += len(choices)
    out["aromatic"], out["in_ring"] = bool(vec[i]), bool(vec[i + 1])
    return out


caffeine = Chem.MolFromSmiles(MOLECULES["caffeine (bitter)"])
for atom in list(caffeine.GetAtoms())[:3]:
    vec = fz.atom_features(atom)
    assert len(vec) == fz.ATOM_FEATURE_DIM
    print(atom.GetIdx(), atom.GetSymbol(), decode_atom(vec))

## 4. Molecule → PyG graph

Nodes are atoms and edges are bonds, with each bond stored in both directions. So
`edge_index` has 2 × (number of bonds) columns.

In [ ]:
for name, smi in MOLECULES.items():
    g = fz.smiles_to_graph(smi)
    n_bonds = Chem.MolFromSmiles(smi).GetNumBonds()
    assert g.edge_index.shape[1] == 2 * n_bonds
    print(
        f"{name:<24} x={tuple(g.x.shape)}  edge_index={tuple(g.edge_index.shape)}  edge_attr={tuple(g.edge_attr.shape)}"
    )

# Edge case: a single ion has no bonds but still gets correctly-shaped empty tensors
print("[Na+]:", fz.smiles_to_graph("[Na+]"))

## 5. Morgan fingerprints and Tanimoto similarity

A Morgan fingerprint (radius 2, 2048 bits) hashes each atom's neighbourhood, up to 2 bonds
out, into a bit vector. Tanimoto similarity is |A ∩ B| / |A ∪ B| over the on-bits. It runs
from 0 (no shared substructures) to 1 (identical fingerprints).

In [ ]:
fps = {name: fz.smiles_to_fingerprint(smi) for name, smi in MOLECULES.items()}
print("shape/dtype:", fps["caffeine (bitter)"].shape, fps["caffeine (bitter)"].dtype)
print("on-bits per molecule:", {n: int(fp.sum()) for n, fp in fps.items()})


def tanimoto(a: np.ndarray, b: np.ndarray) -> float:
    """Tanimoto similarity between two 0/1 fingerprint arrays."""
    union = np.logical_or(a, b).sum()
    return float(np.logical_and(a, b).sum() / union) if union else 0.0

In [ ]:
import pandas as pd

names = list(fps)
sim = pd.DataFrame(
    [[tanimoto(fps[a], fps[b]) for b in names] for a in names],
    index=names,
    columns=names,
)
sim.round(2)

Cross-check our numpy Tanimoto against RDKit's built-in implementation:

In [ ]:
gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
bv = {n: gen.GetFingerprint(Chem.MolFromSmiles(s)) for n, s in MOLECULES.items()}
for a in names:
    for b in names:
        assert abs(DataStructs.TanimotoSimilarity(bv[a], bv[b]) - sim.loc[a, b]) < 1e-9
print("numpy Tanimoto matches RDKit for all pairs")

## Takeaways

- Sucrose and sucralose (sucralose is chlorinated sucrose) should be the most similar
  pair. Structurally unrelated tastants should score low.
- High fingerprint similarity doesn't guarantee the same taste. Sucralose is ~600× sweeter
  than sucrose. This "activity cliff" problem is one reason to try GNNs that can weigh
  individual atoms.
- Similarity also matters for evaluation. A random split puts near-duplicates in both train
  and test, which inflates scores. That's why reported results use a scaffold split.